# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Adjust this path if your repo is stored elsewhere in Drive.
# PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"

In [ ]:
# Install Python dependencies (run once per session)
%pip install -r requirements.txt -q
!python3 -m spacy download en

---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [ ]:
import sys, os

# if PROJECT_ROOT not in sys.path:
#     sys.path.insert(0, PROJECT_ROOT)

# os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [ ]:
from Tools.download import download_mini

download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [ ]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [1]:
from TrainTools.train import train

results = train(
    # -- data paths (must match preprocess outputs) --
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # -- scaled training loop for reaching F1 ~60 --
    num_steps             = 24000,
    checkpoint            = 1500,
    val_num_batches       = 0,
    test_num_batches      = 150,
    accumulate_grad_steps = 4,
    batch_size            = 8,
    seed                  = 42,
    early_stop            = 8,

    # -- combatting overfitting --
    optimizer_name = "adam",
    scheduler_name = "cosine",
    loss_name      = "qa_ce",
    learning_rate  = 1e-3,
    beta1          = 0.8,
    beta2          = 0.999,
    eps            = 1e-7,
    weight_decay   = 3e-5,
    warmup_steps   = 1000,
    grad_clip      = 5.0,
    dropout        = 0.15,
    
    # -- architectural sizing --
    d_model        = 128,
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 1500/1500 [52:36<00:00,  2.10s/it] 


STEP     1500  loss 11.670945

VALID(train) skipped (val_num_batches <= 0)



100%|██████████| 150/150 [00:20<00:00,  7.42it/s]


TEST        loss 7.723595  F1 13.326813  EM 8.250000

Learning rate: [0.0009989294616193018, 0.0009989294616193018]
Checkpoint updated at step 1500 (F1=13.3268, EM=8.2500)


100%|██████████| 1500/1500 [51:50<00:00,  2.07s/it] 


...

TEST        loss 8.842859  F1 68.430884  EM 61.333333

Learning rate: [2.653493525244721e-05, 2.653493525244721e-05]


100%|██████████| 1500/1500 [45:08<00:00,  1.81s/it]


STEP    24000  loss 0.319588

VALID(train) skipped (val_num_batches <= 0)



100%|██████████| 150/150 [00:20<00:00,  7.50it/s]


TEST        loss 9.164884  F1 68.288372  EM 61.166667

Learning rate: [4.277569313094809e-06, 4.277569313094809e-06]
Early stopping triggered.
Training finished.  Best F1: 70.2856  Best EM: 62.5833
Best F1: 70.2856  |  Best EM: 62.5833


---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [2]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
    d_model       = 128,
    loss_name     = "qa_ce",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")


100%|██████████| 1309/1309 [02:54<00:00,  7.50it/s]


TEST  loss 3.718205  F1 70.755265  EM 60.179612  (exact 6299/10467)
F1: 70.7553  |  EM: 60.1796  |  Loss: 3.718205
